# Carbon baseline research: datasets, CatBoost code and reproducible results

Two distinct experiments are provided. Do not pool their rows or compare their error units directly.

## 1. Government dataset — recommended main experiment
City of New York, NYC Open Data: NYC Building Energy and Water Data Disclosure for Local Law 84 2023 to Present (calendar year 2022 onward).
Official page: https://data.cityofnewyork.us/Environment/NYC-Building-Energy-and-Water-Data-Disclosure-for-/5zyy-y8am
Downloaded 5 October 2026: 103,259 rows, 16 selected original columns, report years 2022–2024. This is a documented column subset, not the full original schema. Original values are retained in the downloaded CSV. Source metadata and download query are included.

Target: total location-based GHG emissions, metric tonnes CO2e. Predictors: main and secondary use floor areas, occupancy percentage, construction year, number of buildings, self-selected property type, and borough. Floor areas refer to the listed use types, not a verified sum of all property area. There are no measured production volumes or operating-hour controls in this selected subset.

The code selects the latest year with at least 1,000 raw rows (2024 in this snapshot), filters incomplete/partial metering and reported estimates, removes parent/child reporting and conflicting duplicate property records, and creates property-disjoint train/validation/calibration/test partitions of approximately 60/15/10/15 percent. It tunes CatBoost depth and iteration count on validation log-MAE, compares median and Random Forest models, and calibrates 90% log-residual intervals using calibration properties only. All reported emissions, energy totals, EUI, ENERGY STAR scores, and emission intensities are excluded from predictors. No target-based trimming is done on the test set.

This is SAME-YEAR PEER BENCHMARKING, not future-year forecasting and not a causal no-intervention baseline. A negative or positive peer gap is not proof of additional emission reduction. The data are government-published building-owner submissions, not claimed independently verified sensor measurements. Building performance and carbon policy in New York do not automatically transfer to India.

Suggested provisional paper framing: “Uncertainty-Aware Machine Learning for Building Emission Benchmarking and Simulated Carbon Credit Assessment.” CatBoost and conformal residual calibration are existing methods: novelty needs a focused literature review and comparative research, not merely their inclusion.

## 2. Industrial time-series dataset — supporting experiment
UCI Steel Industry Energy Consumption, 35,040 records, 365 source calendar days in 2018, 96 readings per day. A single South Korean establishment; not multiple establishments.
Official page: https://archive.ics.uci.edu/dataset/851/steel+industry+energy+consumption
Citation: V E, S., Shin, C., & Cho, Y. (2021). Steel Industry Energy Consumption [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C52G8C
License: CC BY 4.0. Retain attribution when sharing.

Target: Usage_kWh. Only calendar/schedule variables are used. Load_Type may be a label derived from electricity use; its provenance does not establish an independent production measurement, so it is excluded. CO2(tCO2) has inconsistent unit labelling in UCI documentation (column name suggests tonnes, units say ppm), so it is excluded. Contemporaneous reactive power and power factors are also excluded to avoid presenting an electrical reconstruction task as a causal baseline. No energy target lags are used, preventing adaptation to a simulated efficiency intervention through those lags.

Train/validation/calibration/test comprise consecutive whole days in approximately 55/15/15/15 proportions. CatBoost depth/iterations are tuned on validation only, then fitted to train+validation. Daily intervals use DAILY calibration residuals; interval-level bounds are not summed to claim daily coverage. Temporal dependence/distribution shift mean classical exchangeability coverage guarantees do not automatically apply.

Credit experiments inject 0%, 10%, and 20% synthetic reductions into held-out observations. Original observations are empirical counterfactuals ONLY inside this simulation. An assumed emission factor of 0.5 kgCO2e/kWh is used in the supplied demonstration; it is NOT an official Indian or Korean factor. Replace it with a documented regional/year factor for real accounting. Credits are illustrative research outputs, not certified tradable units. The 0% scenario quantifies false credit estimation. Daily over/underestimation sums are descriptive metrics, not a guaranteed interval for total annual credits.

## Run locally
```bash
pip install -r requirements.txt
python train_government.py
python train_baseline.py --emission-factor 0.5
```
For energy-only results, omit --emission-factor. Run the scripts from the extracted package or open Carbon_Baseline_Research.ipynb in Colab/Jupyter. The notebook contains the program sources and can download both datasets itself when they are absent. Downloads require internet access. GPU is not required.

Outputs include test MAE/RMSE/R2, tuning tables, feature importance, predictions, calibrated intervals, simulation errors and saved CatBoost models. Feature importance is predictive association, not causal explanation. The government model predicts log1p emissions, so raw-scale predictions approximate a conditional geometric central tendency rather than an unbiased mean. Always report both log and raw-scale evaluation.

## Further work needed for the paper
1. Add pre-intervention energy data and dates of actual efficiency changes; freeze baseline models before intervention.
2. Include independently measured operating hours, occupancy, production, and weather.
3. Evaluate multiple random property splits and future-year temporal tests on government data; the supplied benchmark is one fixed split.
4. Check residual interval coverage by property type and occupancy; overall coverage alone is insufficient.
5. Evaluate missingness, reporting errors, activity changes, and simulated interventions separately.
6. Keep a separate regulatory credit-verification and trading ledger layer. Peer predictions are not emission caps or issued credits.

No results in this package are fabricated; saved results came from executing the accompanying scripts. Exact numeric results may vary with library versions. run_metadata.json records industrial runtime versions and split dates.


In [ ]:
%pip install -q catboost pandas numpy scikit-learn matplotlib
from pathlib import Path
Path("data").mkdir(exist_ok=True)

In [ ]:
import urllib.request, urllib.parse, zipfile, io
steel=Path('data/Steel_industry_data.csv')
if not steel.exists():
    url='https://archive.ics.uci.edu/static/public/851/steel+industry+energy+consumption.zip'
    archive=zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url,timeout=60).read()))
    steel.write_bytes(archive.read('Steel_industry_data.csv'))
fields=['report_year','property_id','parent_property_id','primary_property_type_self','largest_property_use_type_1','_2nd_largest_property_use_1','_3rd_largest_property_use_1','year_built','number_of_buildings','occupancy','borough','total_location_based_ghg','estimated_values_energy','alert_energy_meter_has_less','alert_energy_meter_has_gaps','metered_areas_energy']
nyc=Path('data/NYC_building_emissions_government.csv')
if not nyc.exists():
    url='https://data.cityofnewyork.us/resource/5zyy-y8am.csv?'+urllib.parse.urlencode({'$select':','.join(fields),'$limit':'200000','$order':'report_year,property_id'})
    nyc.write_bytes(urllib.request.urlopen(url,timeout=120).read())
import pandas as pd
print(pd.read_csv(nyc).shape)
print(pd.read_csv(nyc).report_year.value_counts())


In [ ]:
%%writefile train_baseline.py
"""Chronological energy baseline benchmark and explicitly simulated credit study.
Run: python train_baseline.py --emission-factor 0.5
Factor units: kg CO2e/kWh. 0.5 is an illustrative assumption, NOT an official factor.
"""
from pathlib import Path
import argparse, json, math, platform
import numpy as np
import pandas as pd
import sklearn, catboost
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ROOT = Path(__file__).resolve().parent

def metrics(y, p):
    return dict(MAE_kWh=float(mean_absolute_error(y,p)),
                RMSE_kWh=float(np.sqrt(mean_squared_error(y,p))),
                R2=float(r2_score(y,p)))

def finite_quantile(scores, alpha):
    """Split-conformal order statistic; temporal dependence limits guarantees."""
    scores = np.sort(np.asarray(scores, dtype=float))
    k = min(len(scores), math.ceil((len(scores)+1)*(1-alpha)))
    return float(scores[k-1])

def features(d):
    t = d.timestamp
    minute = t.dt.hour*60+t.dt.minute
    dow = t.dt.dayofweek
    # Exclude CO2, reactive power, power factors, current usage and Load_Type.
    # Their contemporaneous/derived nature can inflate baseline performance.
    return pd.DataFrame({
        'minute':minute, 'weekday':dow, 'weekend':(dow>=5).astype(int),
        'hour_sin':np.sin(2*np.pi*minute/1440),
        'hour_cos':np.cos(2*np.pi*minute/1440),
        'week_sin':np.sin(2*np.pi*dow/7),
        'week_cos':np.cos(2*np.pi*dow/7),
    }, index=d.index)

def daily(d, pred):
    return pd.DataFrame({'date':d.day.to_numpy(), 'actual_kWh':d.Usage_kWh.to_numpy(),
                         'predicted_kWh':np.maximum(pred,0)}).groupby('date').sum()

def main():
    parser=argparse.ArgumentParser()
    parser.add_argument('--data',type=Path,default=ROOT/'data/Steel_industry_data.csv')
    parser.add_argument('--output',type=Path,default=ROOT/'results')
    parser.add_argument('--emission-factor',type=float,default=None,
                        help='kg CO2e/kWh; required for simulated carbon results')
    args=parser.parse_args()
    if args.emission_factor is not None and args.emission_factor<=0:
        parser.error('Emission factor must be positive')
    out=args.output; out.mkdir(parents=True,exist_ok=True)
    d=pd.read_csv(args.data)
    d['timestamp']=pd.to_datetime(d['date'],format='%d/%m/%Y %H:%M',errors='raise')
    d=d.sort_values('timestamp').reset_index(drop=True)
    if d.timestamp.duplicated().any() or d.Usage_kWh.isna().any() or (d.Usage_kWh<0).any():
        raise ValueError('Duplicate timestamps, missing target, or negative energy')
    d['day']=d.timestamp.dt.floor('D')
    counts=d.groupby('day').size()
    if not (counts==96).all():
        raise ValueError('Expected 96 readings per source calendar day; inspect incomplete days')
    days=np.sort(d.day.unique()); n=len(days)
    cuts=[0,int(.55*n),int(.70*n),int(.85*n),n]
    parts=[d[d.day.isin(days[cuts[i]:cuts[i+1]])].copy() for i in range(4)]
    train,val,cal,test=parts
    Xtr,Xv,Xc,Xt=[features(p) for p in parts]
    ytr=train.Usage_kWh; yv=val.Usage_kWh
    tuning=[]
    # Selection uses validation only. Calibration and test remain untouched.
    for depth in [4,6,8]:
        for iterations in [400,800]:
            m=CatBoostRegressor(depth=depth,iterations=iterations,learning_rate=.05,
                loss_function='RMSE',random_seed=42,thread_count=4,verbose=False,
                allow_writing_files=False)
            m.fit(Xtr,ytr)
            tuning.append({'depth':depth,'iterations':iterations,
                           'validation_MAE':mean_absolute_error(yv,m.predict(Xv))})
    best=min(tuning,key=lambda x:x['validation_MAE'])
    pd.DataFrame(tuning).to_csv(out/'tuning.csv',index=False)
    fit=pd.concat([train,val]); Xfit=features(fit); yf=fit.Usage_kWh
    models={
        'Ridge':make_pipeline(StandardScaler(),Ridge(alpha=10)),
        'RandomForest':RandomForestRegressor(n_estimators=300,max_depth=12,
            min_samples_leaf=10,random_state=42,n_jobs=4),
        'CatBoost':CatBoostRegressor(depth=best['depth'],iterations=best['iterations'],
            learning_rate=.05,loss_function='RMSE',random_seed=42,thread_count=4,
            verbose=False,allow_writing_files=False),
    }
    rows=[]; daily_predictions={}
    group_means=fit.groupby([fit.timestamp.dt.dayofweek,
                             fit.timestamp.dt.hour*60+fit.timestamp.dt.minute]).Usage_kWh.mean()
    keys=list(zip(test.timestamp.dt.dayofweek,test.timestamp.dt.hour*60+test.timestamp.dt.minute))
    schedule=np.array([group_means.get(k,yf.mean()) for k in keys])
    predictions={'FixedMean':np.full(len(test),yf.mean()),'WeeklySchedule':schedule}
    for name,m in models.items():
        m.fit(Xfit,yf); predictions[name]=np.maximum(m.predict(Xt),0)
    for name,p in predictions.items():
        rows.append({'model':name,'resolution':'15-minute',**metrics(test.Usage_kWh,p)})
        dd=daily(test,p); daily_predictions[name]=dd
        rows.append({'model':name,'resolution':'daily',**metrics(dd.actual_kWh,dd.predicted_kWh)})
    result=pd.DataFrame(rows); result.to_csv(out/'model_metrics.csv',index=False)
    model=models['CatBoost']; model.save_model(str(out/'catboost_baseline.cbm'))
    pd.DataFrame({'feature':Xfit.columns,'importance':model.feature_importances_}).sort_values(
        'importance',ascending=False).to_csv(out/'feature_importance.csv',index=False)
    cd=daily(cal,model.predict(Xc)); td=daily_predictions['CatBoost'].copy()
    # Calibrate on DAY-level residuals, not a sum of interval-level bounds.
    q=finite_quantile(np.abs(cd.actual_kWh-cd.predicted_kWh),alpha=.10)
    td['lower_kWh']=np.maximum(td.predicted_kWh-q,0)
    td['upper_kWh']=td.predicted_kWh+q
    td.to_csv(out/'daily_predictions.csv')
    covered=(td.actual_kWh>=td.lower_kWh)&(td.actual_kWh<=td.upper_kWh)
    uncertainty={'nominal_daily_coverage':.9,'observed_test_daily_coverage':float(covered.mean()),
        'mean_width_kWh':float((td.upper_kWh-td.lower_kWh).mean()),'calibration_days':len(cd),
        'absolute_residual_radius_kWh':q,
        'limitation':'Empirical temporal calibration; no exchangeability guarantee, causal identification, or period-total coverage guarantee.'}
    (out/'uncertainty_metrics.json').write_text(json.dumps(uncertainty,indent=2))
    # Held-out ORIGINAL observations serve as empirical counterfactuals only
    # inside explicitly synthetic efficiency scenarios. No intervention is observed.
    if args.emission_factor is not None:
        factor=args.emission_factor/1000 # kg -> tonnes
        summaries=[]; details=[]
        for fraction in [0,.10,.20]:
            actual=td.actual_kWh*(1-fraction)
            truth=(td.actual_kWh-actual)*factor
            estimates={
                'FixedMean':np.maximum(96*yf.mean()-actual,0)*factor,
                'WeeklySchedule':np.maximum(daily_predictions['WeeklySchedule'].predicted_kWh-actual,0)*factor,
                'CatBoostPoint':np.maximum(td.predicted_kWh-actual,0)*factor,
                'CatBoostConservative':np.maximum(td.lower_kWh-actual,0)*factor,
            }
            for method,estimate in estimates.items():
                summaries.append({'synthetic_reduction_fraction':fraction,'method':method,
                    'daily_MAE_tCO2e':float(np.abs(estimate-truth).mean()),
                    'sum_daily_overestimation_tCO2e':float(np.maximum(estimate-truth,0).sum()),
                    'sum_daily_underestimation_tCO2e':float(np.maximum(truth-estimate,0).sum()),
                    'fraction_days_overcredited':float((estimate>truth+1e-9).mean())})
                details.append(pd.DataFrame({'date':td.index,'synthetic_reduction_fraction':fraction,
                    'method':method,'simulated_actual_kWh':actual.to_numpy(),
                    'simulated_true_reduction_tCO2e':truth.to_numpy(),
                    'simulated_estimate_tCO2e':estimate.to_numpy()}))
        pd.DataFrame(summaries).to_csv(out/'simulation_metrics.csv',index=False)
        pd.concat(details).to_csv(out/'simulated_daily_credits.csv',index=False)
    splits={name:{'first':str(p.day.min().date()),'last':str(p.day.max().date()),
                   'rows':len(p),'days':p.day.nunique()} for name,p in zip(
                       ['train','validation','calibration','test'],parts)}
    manifest={'dataset':'UCI Steel Industry Energy Consumption DOI:10.24432/C52G8C',
        'splits':splits,'selected_catboost':best,'emission_factor_kgCO2e_per_kWh':args.emission_factor,
        'factor_status':'User-supplied assumption; must document authoritative regional/year factor for real accounting',
        'excluded_columns':['CO2(tCO2)','Load_Type','reactive power','power factors'],
        'scope':'Calendar-conditioned electricity baseline. No measured production or intervention labels.',
        'versions':{'python':platform.python_version(),'sklearn':sklearn.__version__,'catboost':catboost.__version__}}
    (out/'run_metadata.json').write_text(json.dumps(manifest,indent=2))
    fig,ax=plt.subplots(figsize=(12,4))
    ax.plot(td.index,td.actual_kWh,label='Observed daily energy',linewidth=1)
    ax.plot(td.index,td.predicted_kWh,label='CatBoost baseline',linewidth=1)
    ax.fill_between(td.index,td.lower_kWh,td.upper_kWh,alpha=.2,label='Empirically calibrated 90% interval')
    ax.set_ylabel('Daily electricity (kWh)');ax.legend();fig.autofmt_xdate();fig.tight_layout()
    fig.savefig(out/'baseline_plot.png',dpi=160);plt.close(fig)
    print(result.to_string(index=False));print(json.dumps(uncertainty,indent=2))
    print('Outputs:',out)

if __name__=='__main__':
    main()


In [ ]:
%%writefile train_government.py
"""NYC government data: property-disjoint annual GHG peer benchmarking.
This is NOT a causal counterfactual or certified credit issuance model.
"""
from pathlib import Path
import json, argparse
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from train_baseline import finite_quantile

ROOT=Path(__file__).resolve().parent

def main():
    p=argparse.ArgumentParser()
    p.add_argument('--data',type=Path,default=ROOT/'data/NYC_building_emissions_government.csv')
    p.add_argument('--year',type=int,default=None)
    args=p.parse_args(); out=ROOT/'government_results';out.mkdir(exist_ok=True)
    raw=pd.read_csv(args.data,dtype=str)
    year=pd.to_numeric(raw.report_year,errors='coerce')
    # Select latest sufficiently populated year; report counts, do not assert completeness.
    counts=year.value_counts(); chosen=args.year or int(counts[counts>=1000].index.max())
    d=raw[year==chosen].copy()
    original=len(d)
    numeric=['largest_property_use_type_1','_2nd_largest_property_use_1',
             '_3rd_largest_property_use_1','year_built','number_of_buildings','occupancy']
    for col in numeric+['total_location_based_ghg']:
        d[col]=pd.to_numeric(d[col].str.replace(',','',regex=False),errors='coerce')
    d=d[d.total_location_based_ghg.notna()&(d.total_location_based_ghg>=0)&d.property_id.notna()]
    d=d[d.largest_property_use_type_1>0]
    # Remove parent/child campuses to reduce overlapping reporting boundaries.
    d=d[d.parent_property_id.fillna('Not Available').isin(['Not Available','Not Applicable: Standalone Property','0',''])]
    for col in ['alert_energy_meter_has_less','alert_energy_meter_has_gaps']:
        d=d[~d[col].fillna('').str.lower().isin(['yes','true'])]
    # Keep only fully metered properties. Do not silently interpret partial coverage.
    d=d[d.metered_areas_energy.fillna('').str.lower().isin(['whole building','whole property'])]
    d=d[~d.estimated_values_energy.fillna('').str.lower().isin(['yes','true'])]
    d=d.drop_duplicates().copy()
    # Conflicting repeated property/year records are excluded, not averaged.
    d=d[~d.property_id.duplicated(keep=False)].reset_index(drop=True)
    for c in numeric:
        d.loc[d[c]<0,c]=np.nan
    d.loc[(d.occupancy>100), 'occupancy']=np.nan
    d.loc[(d.year_built<1600)|(d.year_built>chosen),'year_built']=np.nan
    nums=numeric
    cats=['primary_property_type_self','borough']
    X=d[nums+cats].copy()
    X[cats]=X[cats].fillna('Unknown').astype(str)
    y=np.log1p(d.total_location_based_ghg.to_numpy())
    # Whole properties are assigned to one partition only. Same year, cross-sectional.
    idx=np.arange(len(d));groups=d.property_id
    def split(ids, fraction, seed):
        a,b=next(GroupShuffleSplit(n_splits=1,test_size=fraction,random_state=seed).split(
            ids,groups=groups.iloc[ids]))
        return ids[a],ids[b]
    remain,test=split(idx,.15,42); remain,cal=split(remain,.10/.85,43)
    train,val=split(remain,.15/.75,44)
    assert not set(groups.iloc[train])&set(groups.iloc[test])
    med=X.iloc[train][nums].median().fillna(0)
    X[nums]=X[nums].fillna(med)
    tuning=[]
    for depth in [4,6,8]:
        for it in [400,800]:
            m=CatBoostRegressor(iterations=it,depth=depth,learning_rate=.05,loss_function='RMSE',
                random_seed=42,verbose=False,thread_count=4,allow_writing_files=False)
            m.fit(X.iloc[train],y[train],cat_features=cats)
            tuning.append({'depth':depth,'iterations':it,
                           'validation_log_MAE':mean_absolute_error(y[val],m.predict(X.iloc[val]))})
    best=min(tuning,key=lambda r:r['validation_log_MAE'])
    fit=np.concatenate([train,val])
    # The same train-derived feature imputation is retained in all partitions.
    m=CatBoostRegressor(iterations=best['iterations'],depth=best['depth'],learning_rate=.05,
        loss_function='RMSE',random_seed=42,verbose=False,thread_count=4,allow_writing_files=False)
    m.fit(X.iloc[fit],y[fit],cat_features=cats)
    rf=make_pipeline(ColumnTransformer([
        ('nums',SimpleImputer(strategy='median'),nums),
        ('cats',OneHotEncoder(handle_unknown='ignore',sparse_output=False),cats)]),
        RandomForestRegressor(n_estimators=300,min_samples_leaf=5,max_depth=18,n_jobs=4,random_state=42))
    rf.fit(X.iloc[fit],y[fit])
    preds={'Median':np.full(len(test),np.median(y[fit])),
           'RandomForest':rf.predict(X.iloc[test]),'CatBoost':m.predict(X.iloc[test])}
    rows=[]; actual=np.expm1(y[test])
    for name,logpred in preds.items():
        pred=np.maximum(np.expm1(logpred),0)
        rows.append({'model':name,'MAE_tCO2e':mean_absolute_error(actual,pred),
          'RMSE_tCO2e':np.sqrt(mean_squared_error(actual,pred)),
          'R2':r2_score(actual,pred),'MAE_log1p':mean_absolute_error(y[test],logpred)})
    q=finite_quantile(np.abs(y[cal]-m.predict(X.iloc[cal])),.1)
    lp=preds['CatBoost'];lo=np.maximum(np.expm1(lp-q),0);hi=np.maximum(np.expm1(lp+q),0)
    result=d.iloc[test][['property_id','primary_property_type_self','total_location_based_ghg']].copy()
    result['peer_baseline_tCO2e']=np.maximum(np.expm1(lp),0)
    result['lower_tCO2e']=lo;result['upper_tCO2e']=hi
    result['gap_to_peer_baseline_tCO2e']=result.peer_baseline_tCO2e-actual
    result.to_csv(out/'government_test_predictions.csv',index=False)
    simulations=[]
    for fraction in [0,.1,.2]:
        simulated_actual=actual*(1-fraction)
        truth=actual-simulated_actual
        for name,baseline in {'Median':np.full(len(test),np.expm1(np.median(y[fit]))),
                             'CatBoostPoint':result.peer_baseline_tCO2e.to_numpy(),
                             'CatBoostConservative':lo}.items():
            estimate=np.maximum(baseline-simulated_actual,0)
            simulations.append({'synthetic_reduction_fraction':fraction,'method':name,
                'MAE_tCO2e':float(np.abs(estimate-truth).mean()),
                'sum_overestimation_tCO2e':float(np.maximum(estimate-truth,0).sum()),
                'sum_underestimation_tCO2e':float(np.maximum(truth-estimate,0).sum()),
                'fraction_properties_overcredited':float((estimate>truth+1e-9).mean())})
    pd.DataFrame(simulations).to_csv(out/'government_simulation_metrics.csv',index=False)
    pd.DataFrame(rows).to_csv(out/'government_model_metrics.csv',index=False)
    pd.DataFrame(tuning).to_csv(out/'government_tuning.csv',index=False)
    pd.DataFrame({'feature':X.columns,'importance':m.feature_importances_}).sort_values(
        'importance',ascending=False).to_csv(out/'government_feature_importance.csv',index=False)
    m.save_model(str(out/'government_catboost.cbm'))
    metadata={'source':'City of New York NYC Open Data dataset 5zyy-y8am',
      'selected_report_year':chosen,'downloaded_rows':len(raw),'year_rows_before_cleaning':original,
      'clean_rows':len(d),'partition_rows':{k:len(v) for k,v in zip(
          ['train','validation','calibration','test'],[train,val,cal,test])},
      'nominal_coverage':.9,'test_coverage':float(((actual>=lo)&(actual<=hi)).mean()),
      'mean_interval_width_tCO2e':float(np.mean(hi-lo)),'selected_model':best,
      'features':list(X.columns),'imputation_values':med.to_dict(),
      'scope':'Property-disjoint, same-year peer emissions benchmark; NOT a verified causal baseline or carbon credits.',
      'limitations':['Self-reported government-published records','Incomplete activity/weather controls',
                    'US building results do not validate Indian establishments',
                    'No future-year validation; latest available year is not claimed complete',
                    'Log-space intervals assume sufficiently comparable calibration/test properties']}
    (out/'government_run_metadata.json').write_text(json.dumps(metadata,indent=2))
    print(pd.DataFrame(rows).to_string(index=False));print(json.dumps(metadata,indent=2))

if __name__=='__main__': main()


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,"train_government.py"],check=True)

In [ ]:
pd.read_csv("government_results/government_model_metrics.csv")

In [ ]:
subprocess.run([sys.executable,"train_baseline.py","--emission-factor","0.5"],check=True)

In [ ]:
pd.read_csv("results/model_metrics.csv")

In [ ]:
pd.read_csv("results/simulation_metrics.csv")

In [ ]:
from IPython.display import Image, display
display(Image(filename="results/baseline_plot.png"))